# AbuseRing Sentinel - Graph Analysis (Phase 2)

Exploratory visualisation of the **heterogeneous relationship graph** and the
**user projection** built from the synthetic Phase 1 dataset.

**All data is synthetic.** Community detection is **label-free** - it never sees
abuse labels. Where a ground-truth abuse ring is drawn below, it is shown *for
exploratory analysis only*; the graph algorithms do not know it is an abuse
ring. No model is trained here and no performance metric is produced.

Prerequisites:

```bash
python -m src.generators.pipeline   # Phase 1 data
python -m src.features.pipeline     # Phase 2 features + split graph views
python -m src.graph.validate        # Phase 2 reports
```

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.validation.loader import load_tables
from src.graph.builder import build_graph, nodes_of_type
from src.graph.projection import build_user_projection
from src.graph.communities import detect_communities
from src.graph.schema import NodeType

tables = load_tables(ROOT / 'data' / 'processed')
users, txns = tables['users'], tables['transactions']
label = users.set_index('user_id')['is_abuse_account'].to_dict()
ring_of = users.set_index('user_id')['ring_id'].to_dict()
ring_type_of = users.set_index('user_id')['ring_type'].to_dict()

## 1. Graph size statistics

In [ ]:
hetero = build_graph(tables)
proj = build_user_projection(txns).graph
print('heterogeneous graph:', hetero.number_of_nodes(), 'nodes,', hetero.number_of_edges(), 'edges')
print('user projection    :', proj.number_of_nodes(), 'nodes,', proj.number_of_edges(), 'edges')
for nt in NodeType:
    print(f'  {nt.value:<20}: {len(nodes_of_type(hetero, nt))}')

## 2. Degree distribution (user projection)

In [ ]:
degs = [d for _, d in proj.degree()]
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(degs, bins=50)
ax.set_title('User-projection degree distribution')
ax.set_xlabel('degree'); ax.set_ylabel('users'); ax.set_yscale('log')
plt.tight_layout(); plt.show()

## 3. Community size distribution (Louvain, label-free)

In [ ]:
comm = detect_communities(proj)
sizes = comm.groupby('community_id')['community_size'].first()
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(sizes, bins=40)
ax.set_title('Community size distribution')
ax.set_xlabel('community size'); ax.set_ylabel('communities')
plt.tight_layout(); plt.show()
print('communities:', comm['community_id'].nunique())

In [ ]:
def draw_user_subgraph(user_ids, title, highlight_abuse=True):
    """Draw the projection induced on a set of users."""
    sub = proj.subgraph(user_ids)
    pos = nx.spring_layout(sub, seed=42, k=0.5)
    colors = ['#d62728' if label.get(n) else '#1f77b4' for n in sub.nodes()]
    fig, ax = plt.subplots(figsize=(7, 5))
    nx.draw_networkx_edges(sub, pos, alpha=0.3, ax=ax)
    nx.draw_networkx_nodes(sub, pos, node_color=colors, node_size=120, ax=ax)
    ax.set_title(title)
    ax.axis('off')
    # red = ground-truth abuse account, blue = legitimate (for exploration only)
    plt.tight_layout(); plt.show()

## 4. Example legitimate high-connectivity group

A large legitimate shared-infrastructure group (e.g. college/hostel). High
connectivity, **no** abuse - illustrates why shared IP/device alone is not fraud.

In [ ]:
# Find a large community that is (by ground truth, for display only) mostly legit.
cdf = comm.copy(); cdf['abuse'] = cdf['user_id'].map(label).astype(int)
stats = cdf.groupby('community_id').agg(size=('user_id','size'), abuse=('abuse','mean'))
legit_comm = stats[(stats['size'].between(15, 60)) & (stats['abuse'] < 0.05)].sort_values('size', ascending=False)
if len(legit_comm):
    cid = legit_comm.index[0]
    members = cdf[cdf['community_id'] == cid]['user_id'].tolist()
    draw_user_subgraph(members, f'Legitimate high-connectivity community #{cid} (blue=legit)')
else:
    print('No matching community at this scale.')

## 5. Example abuse ring

**Ground-truth abuse ring shown for exploratory analysis.** The graph/community
algorithms do not know this is an abuse ring.

In [ ]:
direct_rings = [r for r in users[users['is_abuse_account']]['ring_id'].unique()
                if r and ring_type_of.get(users[users.ring_id==r].user_id.iloc[0]) == 'direct_infrastructure']
abuse_users = users[users['ring_id'] == direct_rings[0]]['user_id'].tolist()
draw_user_subgraph(abuse_users, f'Ground-truth abuse ring {direct_rings[0]} (direct_infrastructure) - red=abuse')

## 6. Example indirect multi-hop structure

**Ground-truth abuse ring shown for exploratory analysis.** An indirect ring is
connected through *chains* of shared entities - no single entity links everyone.

In [ ]:
indirect = [r for r in users[users['is_abuse_account']]['ring_id'].unique()
            if r and ring_type_of.get(users[users.ring_id==r].user_id.iloc[0]) == 'indirect_multihop']
if indirect:
    ring_users = users[users['ring_id'] == indirect[0]]['user_id'].tolist()
    draw_user_subgraph(ring_users, f'Indirect multi-hop ring {indirect[0]} - red=abuse')
    sub = proj.subgraph(ring_users)
    print('ring users:', len(ring_users), '| projection edges among them:', sub.number_of_edges())
    print('connected components within ring:', nx.number_connected_components(sub))
else:
    print('No indirect ring found.')